# Player Lifecycle & Cross-Sector Analytics

Комплексний ноутбук для аналізу поведінки, прогнозування та канальної ефективності гравців вересня.


## 1. План дослідження
1. Імпорт бібліотек та перевірка залежностей
2. Завантаження/імітація даних і первинна діагностика
3. Препроцесинг та побудова повної сітки (user × date × sector)
4. Ознаки: RFM, лаги, rolling, behavioral, statistical, contextual
5. Поведінковий аналіз та матриці переходів
6. Кластеризація користувачів
7. Прогностичні моделі (класифікація/регресія/активація)
8. Sequence mining та аномалії
9. Канальна ефективність
10. Бізнес-інсайти


## 2. Імпорт бібліотек


In [ ]:
import importlib
import warnings
warnings.filterwarnings('ignore')

REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'seaborn': 'seaborn',
    'matplotlib': 'matplotlib',
    'scipy': 'scipy',
    'sklearn': 'sklearn',
    'lightgbm': 'lightgbm',
    'mlxtend': 'mlxtend',
    'hmmlearn': 'hmmlearn',
    'ruptures': 'ruptures',
    'shap': 'shap'
}
missing = [pkg for pkg, module in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print('⚠️ Відсутні пакети:', ', '.join(missing))
    print('Встановіть за потреби через %pip install ' + ' '.join(missing))
else:
    print('✅ Усі пакети наявні')


In [ ]:
from pathlib import Path
from typing import List, Dict

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import entropy as scipy_entropy
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, classification_report, roc_auc_score, average_precision_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import IsolationForest

try:
    import lightgbm as lgb
except Exception:
    lgb = None

try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import fpgrowth
except Exception:
    TransactionEncoder = None
    fpgrowth = None

try:
    from hmmlearn import hmm
except Exception:
    hmm = None

try:
    import ruptures as rpt
except Exception:
    rpt = None

try:
    import shap
except Exception:
    shap = None

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(7)


## 3. Завантаження даних
- Якщо існує `data/transactions.csv`, читаємо його.
- Інакше генеруємо синтетичний датасет (30 днів, 200 користувачів).


In [ ]:
DATA_PATH = Path('data/transactions.csv')
SECTORS = ['virtual', 'other', 'tv']
CHANNELS = ['organic', 'paid_search', 'paid_social', 'affiliate']

if DATA_PATH.exists():
    raw_df = pd.read_csv(DATA_PATH)
    origin = 'source'
else:
    start_date = pd.Timestamp('2024-09-01')
    dates = pd.date_range(start_date, periods=30, freq='D')
    users = np.arange(10_000, 10_200)
    rows = []
    for user in users:
        channel = np.random.choice(CHANNELS, p=[0.35, 0.25, 0.25, 0.15])
        active_days = np.random.choice(dates, size=np.random.randint(6, 18), replace=False)
        for day in active_days:
            for sector in SECTORS:
                engage = (sector == 'virtual') or (np.random.rand() < 0.45)
                if not engage:
                    continue
                spend = np.random.gamma(shape=2.0, scale=5.0)
                if np.random.rand() < 0.25:
                    spend = 0.0
                rows.append({
                    'user_id': user,
                    'date': day.strftime('%Y-%m-%d'),
                    'spend': round(float(spend), 2),
                    'sector': sector,
                    'acquisition_channel': channel
                })
    raw_df = pd.DataFrame(rows)
    origin = 'synthetic'

print('Dataset origin:', origin)
print('Rows:', len(raw_df))
raw_df.head()


In [ ]:
raw_df.info()
raw_df.describe(include='all')


## 4. Препроцесинг та побудова сітки


In [ ]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['sector'] = df['sector'].str.lower().str.strip().replace({'sports': 'other', 'tv-show': 'tv'})
    df.loc[~df['sector'].isin(SECTORS), 'sector'] = 'other'
    df['acquisition_channel'] = df['acquisition_channel'].fillna('unknown').str.lower()
    df['spend'] = df['spend'].fillna(0.0).astype(float)
    return (
        df.groupby(['user_id', 'date', 'sector', 'acquisition_channel'], as_index=False)
          .agg(spend=('spend', 'sum'))
    )


def primary_channel(df: pd.DataFrame) -> pd.DataFrame:
    order = {'organic': 0, 'affiliate': 1, 'paid_search': 2, 'paid_social': 3}
    df = df.assign(rank=df['acquisition_channel'].map(order).fillna(9))
    return df.sort_values(['user_id', 'rank', 'date']).groupby('user_id', as_index=False).first()[['user_id', 'acquisition_channel']]


def build_grid(transactions: pd.DataFrame) -> pd.DataFrame:
    min_date, max_date = transactions['date'].min(), transactions['date'].max()
    date_range = pd.date_range(min_date, max_date, freq='D')
    users = transactions['user_id'].unique()
    channels = primary_channel(transactions)
    multi_index = pd.MultiIndex.from_product([users, date_range, SECTORS], names=['user_id', 'date', 'sector'])
    base = multi_index.to_frame(index=False).merge(channels, on='user_id', how='left')
    merged = base.merge(transactions, on=['user_id', 'date', 'sector', 'acquisition_channel'], how='left')
    merged['spend'] = merged['spend'].fillna(0.0)
    return merged.sort_values(['user_id', 'date', 'sector']).reset_index(drop=True)


def daily_summary(grid: pd.DataFrame) -> pd.DataFrame:
    daily = grid.groupby(['user_id', 'date'], as_index=False).agg(total_spend=('spend', 'sum'))
    daily['is_active'] = (daily['total_spend'] > 0).astype(int)
    gap_records = []
    for user, user_df in daily.groupby('user_id'):
        user_df = user_df.sort_values('date').reset_index(drop=True)
        next_active = None
        gaps = []
        for idx in range(len(user_df) - 1, -1, -1):
            current = user_df.loc[idx, 'date']
            if next_active is None:
                gaps.append(np.nan)
            else:
                gaps.append((next_active - current).days)
            if user_df.loc[idx, 'is_active']:
                next_active = current
        user_df['gap_days'] = list(reversed(gaps))
        gap_records.append(user_df)
    result = pd.concat(gap_records, ignore_index=True)
    result['churn_flag'] = (result['gap_days'] > 7).astype(int)
    return result

transactions = preprocess(raw_df)
full_grid = build_grid(transactions)
daily_metrics = daily_summary(full_grid)

print('Transactions:', len(transactions), 'Full grid:', len(full_grid))
full_grid.head()


## 5. Ознаки


In [ ]:
def compute_features(grid: pd.DataFrame, daily: pd.DataFrame) -> pd.DataFrame:
    df = grid.copy()
    df = df.sort_values(['user_id', 'sector', 'date'])
    df['spend_lag_1'] = df.groupby(['user_id', 'sector'])['spend'].shift(1).fillna(0.0)
    df['spend_lag_7'] = df.groupby(['user_id', 'sector'])['spend'].shift(7).fillna(0.0)
    df['roll_sum_3'] = df.groupby(['user_id', 'sector'])['spend'].transform(lambda s: s.rolling(3, min_periods=1).sum())
    df['roll_mean_7'] = df.groupby(['user_id', 'sector'])['spend'].transform(lambda s: s.rolling(7, min_periods=1).mean())
    df['roll_std_7'] = df.groupby(['user_id', 'sector'])['spend'].transform(lambda s: s.rolling(7, min_periods=1).std().fillna(0.0))

    pivot = df.pivot_table(index=['user_id', 'date'], columns='sector', values='spend', fill_value=0.0)
    for s in SECTORS:
        if s not in pivot.columns:
            pivot[s] = 0.0
    pivot['virtual_other_ratio'] = pivot['virtual'] / (pivot['other'] + 1e-6)
    pivot['virtual_tv_ratio'] = pivot['virtual'] / (pivot['tv'] + 1e-6)
    pivot['other_tv_ratio'] = pivot['other'] / (pivot['tv'] + 1e-6)
    ratio_df = pivot.reset_index()

    rfm_base = df[df['spend'] > 0]
    max_date = df['date'].max()
    rfm = (
        rfm_base.groupby(['user_id', 'sector'])
                .agg(
                    recency=('date', lambda x: (max_date - x.max()).days),
                    frequency=('date', 'nunique'),
                    monetary=('spend', 'sum')
                )
                .reset_index()
    )
    rfm_pivot = rfm.pivot(index='user_id', columns='sector', values=['recency', 'frequency', 'monetary']).fillna(0)
    rfm_pivot.columns = [f"{m}_{s}" for m, s in rfm_pivot.columns]
    rfm_pivot = rfm_pivot.reset_index()

    behavior = []
    for user, user_daily in daily.groupby('user_id'):
        user_daily = user_daily.sort_values('date')
        streak = 0
        streaks = []
        cooldowns = []
        last_active = None
        for _, row in user_daily.iterrows():
            if row['is_active']:
                streak += 1
                if last_active is not None:
                    cooldowns.append((row['date'] - last_active).days)
                last_active = row['date']
            else:
                streak = 0
            streaks.append(streak)
        behavior.append({
            'user_id': user,
            'days_active': int(user_daily['is_active'].sum()),
            'longest_streak': int(max(streaks) if streaks else 0),
            'median_cooldown': float(np.median(cooldowns)) if cooldowns else np.nan,
            'zero_share': 1 - user_daily['is_active'].mean(),
            'avg_gap': user_daily['gap_days'].mean(skipna=True),
            'max_gap': user_daily['gap_days'].max(skipna=True)
        })
    behavior_df = pd.DataFrame(behavior)

    stats = []
    for user, user_df in df.groupby('user_id'):
        spend_by_sector = user_df.groupby('sector')['spend'].sum().reindex(SECTORS, fill_value=0.0)
        total = spend_by_sector.sum()
        probs = (spend_by_sector / total).replace([np.inf, np.nan], 0)
        stats.append({
            'user_id': user,
            'gini_spend': float(1 - np.sum(np.square(probs))) if total > 0 else 0.0,
            'entropy_spend': float(scipy_entropy(probs + 1e-9, base=2)) if total > 0 else 0.0
        })
    stats_df = pd.DataFrame(stats)

    df = df.merge(ratio_df, on=['user_id', 'date'], how='left')
    df = df.merge(daily[['user_id', 'date', 'gap_days', 'churn_flag']], on=['user_id', 'date'], how='left')
    df['gap_days'] = df['gap_days'].fillna(0)
    df['churn_flag'] = df['churn_flag'].fillna(0).astype(int)

    day_ctx = df[['user_id', 'date']].copy()
    day_ctx['day_of_week'] = df['date'].dt.day_name()
    day_ctx['weekday_idx'] = df['date'].dt.weekday
    day_ctx['is_weekend'] = day_ctx['weekday_idx'].isin([5, 6]).astype(int)

    enriched = df.merge(day_ctx, on=['user_id', 'date'], how='left')
    user_features = behavior_df.merge(stats_df, on='user_id', how='left').merge(rfm_pivot, on='user_id', how='left')
    return enriched, user_features

feature_daily, user_features = compute_features(full_grid, daily_metrics)
feature_daily.head()


## 6. Матриці переходів та динаміка витрат


In [ ]:
def transition_matrix(grid: pd.DataFrame, weighted: bool = False) -> pd.DataFrame:
    records = []
    for user, user_df in grid.groupby('user_id'):
        daily = user_df.pivot_table(index='date', columns='sector', values='spend', fill_value=0.0)
        daily['state'] = np.where(daily.sum(axis=1) == 0, 'pause', daily[SECTORS].idxmax(axis=1))
        daily['weight'] = daily.sum(axis=1)
        states = daily[['state', 'weight']].reset_index()
        for i in range(len(states) - 1):
            w = states.loc[i + 1, 'weight'] if weighted else 1
            records.append((states.loc[i, 'state'], states.loc[i + 1, 'state'], w))
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records, columns=['from', 'to', 'value'])
    return df.pivot_table(index='from', columns='to', values='value', aggfunc='sum', fill_value=0.0)

trans_counts = transition_matrix(full_grid, weighted=False)
trans_weighted = transition_matrix(full_grid, weighted=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
if not trans_counts.empty:
    sns.heatmap(trans_counts, annot=True, fmt='.0f', cmap='Blues', ax=axes[0])
    axes[0].set_title('Transition counts')
else:
    axes[0].set_visible(False)
if not trans_weighted.empty:
    sns.heatmap(trans_weighted, annot=True, fmt='.1f', cmap='Purples', ax=axes[1])
    axes[1].set_title('Weighted transitions')
else:
    axes[1].set_visible(False)
plt.show()

rolling = (full_grid.groupby(['date', 'sector'])['spend'].sum().reset_index())
rolling['roll_mean_7'] = rolling.groupby('sector')['spend'].transform(lambda s: s.rolling(7, min_periods=1).mean())
plt.figure(figsize=(12, 6))
for sector in SECTORS:
    subset = rolling[rolling['sector'] == sector]
    plt.plot(subset['date'], subset['roll_mean_7'], label=f'{sector} 7d mean')
plt.legend(); plt.title('7-day rolling spend by sector'); plt.show()

cooldown_stats = daily_metrics.groupby('user_id').agg(
    avg_gap=('gap_days', 'mean'),
    cooldown_75=('gap_days', lambda x: np.nanpercentile(x, 75)),
    churn_rate=('churn_flag', 'mean'),
    active_days=('is_active', 'sum')
)
cooldown_stats.describe()


## 7. Кластеризація


In [ ]:
cluster_input = user_features.fillna(0).set_index('user_id')
scaler = StandardScaler()
scaled = scaler.fit_transform(cluster_input)

sil_scores = {}
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init='auto')
    sil_scores[k] = silhouette_score(scaled, model.fit_predict(scaled))

best_k = max(sil_scores, key=sil_scores.get)
km = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
cluster_labels = km.fit_predict(scaled)
cluster_input['cluster'] = cluster_labels

pca = PCA(n_components=2).fit_transform(scaled)
cluster_input['pc1'] = pca[:, 0]
cluster_input['pc2'] = pca[:, 1]

plt.figure(figsize=(9, 7))
sns.scatterplot(data=cluster_input, x='pc1', y='pc2', hue='cluster', palette='viridis')
plt.title('PCA projection of clusters')
plt.show()

cluster_summary = cluster_input.groupby('cluster').agg(
    users=('pc1', 'count'),
    avg_days_active=('days_active', 'mean'),
    avg_zero_share=('zero_share', 'mean'),
    avg_virtual_spend=('monetary_virtual', 'mean'),
    avg_tv_spend=('monetary_tv', 'mean'),
    avg_entropy=('entropy_spend', 'mean')
)
cluster_summary


## 8. Прогностичні моделі


In [ ]:
def modeling_frame(df: pd.DataFrame) -> pd.DataFrame:
    pivot = df.pivot_table(index=['user_id', 'date'], columns='sector', values='spend', fill_value=0.0).reset_index()
    features = df.drop_duplicates(['user_id', 'date'])[['user_id', 'date', 'virtual_other_ratio', 'virtual_tv_ratio', 'other_tv_ratio', 'gap_days', 'churn_flag']]
    merged = pivot.merge(features, on=['user_id', 'date'], how='left')
    merged['next_date'] = merged.groupby('user_id')['date'].shift(-1)
    for sector in SECTORS:
        merged[f'next_{sector}'] = merged.groupby('user_id')[sector].shift(-1)
    merged['next_sector'] = merged[[f'next_{s}' for s in SECTORS]].idxmax(axis=1).str.replace('next_', '')
    merged['next_spend'] = merged[[f'next_{s}' for s in SECTORS]].sum(axis=1)
    merged = merged.dropna(subset=['next_date'])
    merged = pd.get_dummies(merged, columns=['churn_flag'], drop_first=False)
    merged = pd.get_dummies(merged, columns=[], drop_first=False)
    return merged

model_df = modeling_frame(feature_daily)
model_df.head()


In [ ]:
def train_classifier(df: pd.DataFrame):
    if lgb is None:
        print('LightGBM недоступний')
        return None
    X = df.drop(columns=['user_id', 'date', 'next_date', 'next_sector', 'next_spend'])
    y = df['next_sector']
    mask = y != 'nan'
    X, y = X[mask], y[mask]
    splits = TimeSeriesSplit(n_splits=3)
    reports = []
    for fold, (train_idx, test_idx) in enumerate(splits.split(X), 1):
        train_x, test_x = X.iloc[train_idx], X.iloc[test_idx]
        train_y, test_y = y.iloc[train_idx], y.iloc[test_idx]
        lgb_train = lgb.Dataset(train_x, label=train_y)
        lgb_val = lgb.Dataset(test_x, label=test_y, reference=lgb_train)
        params = {
            'objective': 'multiclass',
            'num_class': len(y.unique()),
            'learning_rate': 0.05,
            'metric': 'multi_logloss',
            'num_leaves': 31,
            'verbose': -1
        }
        model = lgb.train(params, lgb_train, valid_sets=[lgb_val], num_boost_round=400, early_stopping_rounds=40, verbose_eval=False)
        preds = np.argmax(model.predict(test_x), axis=1)
        print(f'Fold {fold}
', classification_report(test_y, preds))
        reports.append(model)
    return reports[-1] if reports else None


def train_regressor(df: pd.DataFrame):
    if lgb is None:
        print('LightGBM недоступний')
        return None
    X = df.drop(columns=['user_id', 'date', 'next_date', 'next_sector', 'next_spend'])
    y = df['next_spend']
    splits = TimeSeriesSplit(n_splits=3)
    metrics = []
    for fold, (train_idx, test_idx) in enumerate(splits.split(X), 1):
        train_x, test_x = X.iloc[train_idx], X.iloc[test_idx]
        train_y, test_y = y.iloc[train_idx], y.iloc[test_idx]
        lgb_train = lgb.Dataset(train_x, label=train_y)
        lgb_val = lgb.Dataset(test_x, label=test_y, reference=lgb_train)
        params = {
            'objective': 'regression',
            'metric': ['l1', 'l2'],
            'learning_rate': 0.05,
            'num_leaves': 31,
            'feature_fraction': 0.9,
            'verbose': -1
        }
        model = lgb.train(params, lgb_train, valid_sets=[lgb_val], num_boost_round=400, early_stopping_rounds=40, verbose_eval=False)
        preds = model.predict(test_x)
        mae = mean_absolute_error(test_y, preds)
        rmse = mean_squared_error(test_y, preds, squared=False)
        metrics.append({'fold': fold, 'MAE': mae, 'RMSE': rmse})
        print(f'Fold {fold}: MAE={mae:.3f}, RMSE={rmse:.3f}')
    return metrics

clf_model = train_classifier(model_df) if lgb is not None else None
reg_metrics = train_regressor(model_df) if lgb is not None else None
reg_metrics


In [ ]:
def activation_model(df: pd.DataFrame):
    if lgb is None:
        print('LightGBM недоступний')
        return None
    df = df.copy()
    df['activate_other_tv'] = (df['next_other'] + df['next_tv'] > 0).astype(int)
    X = df.drop(columns=['user_id', 'date', 'next_date', 'next_sector', 'next_spend', 'activate_other_tv'])
    y = df['activate_other_tv']
    splits = TimeSeriesSplit(n_splits=3)
    metrics = []
    for fold, (train_idx, test_idx) in enumerate(splits.split(X), 1):
        train_x, test_x = X.iloc[train_idx], X.iloc[test_idx]
        train_y, test_y = y.iloc[train_idx], y.iloc[test_idx]
        train_set = lgb.Dataset(train_x, label=train_y)
        val_set = lgb.Dataset(test_x, label=test_y, reference=train_set)
        params = {'objective': 'binary', 'metric': ['auc', 'binary_logloss'], 'learning_rate': 0.05, 'num_leaves': 31}
        model = lgb.train(params, train_set, valid_sets=[val_set], num_boost_round=300, early_stopping_rounds=30, verbose_eval=False)
        preds = model.predict(test_x)
        metrics.append({'fold': fold, 'ROC-AUC': roc_auc_score(test_y, preds), 'PR-AUC': average_precision_score(test_y, preds)})
    return pd.DataFrame(metrics)

activation_stats = activation_model(model_df) if lgb is not None else None
activation_stats


## 9. Sequence mining та аномалії


In [ ]:
def build_sequences(grid: pd.DataFrame) -> List[List[str]]:
    sequences = []
    for _, user_df in grid.groupby('user_id'):
        daily = user_df.pivot_table(index='date', columns='sector', values='spend', fill_value=0.0)
        sequence = daily.apply(lambda row: 'pause' if row.sum() == 0 else row[SECTORS].idxmax(), axis=1).tolist()
        sequences.append(sequence)
    return sequences

sequences = build_sequences(full_grid)
print('Total sequences:', len(sequences))

if TransactionEncoder is not None and fpgrowth is not None:
    te = TransactionEncoder()
    tx = te.fit_transform([list(dict.fromkeys(seq)) for seq in sequences])
    freq = fpgrowth(pd.DataFrame(tx, columns=te.columns_), min_support=0.1, use_colnames=True)
    freq.sort_values('support', ascending=False).head()
else:
    print('mlxtend недоступний')

if hmm is not None:
    mapping = {'virtual': 0, 'other': 1, 'tv': 2, 'pause': 3}
    concatenated = []
    lengths = []
    for seq in sequences:
        concatenated.extend([mapping[s] for s in seq])
        lengths.append(len(seq))
    model = hmm.MultinomialHMM(n_components=4, random_state=42, n_iter=100)
    model.fit(np.array(concatenated).reshape(-1, 1), lengths)
    print('HMM transitions:
', model.transmat_)
else:
    print('hmmlearn недоступний')

if rpt is not None:
    total_spend = full_grid.groupby('date')['spend'].sum().reset_index()
    algo = rpt.Pelt(model='rbf').fit(total_spend['spend'].values)
    cp = algo.predict(pen=5)
    print('Change points:', cp)
else:
    print('ruptures недоступний')

iso = IsolationForest(contamination=0.02, random_state=42)
user_daily = full_grid.groupby(['user_id', 'date'])['spend'].sum().reset_index()
user_daily['anomaly_score'] = iso.fit_predict(user_daily[['spend']])
user_daily[user_daily['anomaly_score'] == -1].head()


## 10. Канальна ефективність


In [ ]:
channel_perf = feature_daily.groupby('acquisition_channel').agg(
    users=('user_id', 'nunique'),
    active_share=('spend', lambda x: (x > 0).mean()),
    avg_spend=('spend', 'mean'),
    avg_gap=('gap_days', 'mean'),
    median_gap=('gap_days', 'median'),
    virtual_focus=('virtual', 'mean') if 'virtual' in feature_daily.columns else ('spend', 'mean'),
    ratio_virtual_other=('virtual_other_ratio', 'mean'),
    ratio_virtual_tv=('virtual_tv_ratio', 'mean')
).reset_index()
channel_perf


In [ ]:
cluster_mix = cluster_input.reset_index().merge(primary_channel(transactions), on='user_id', how='left')
cluster_mix = cluster_mix.groupby(['acquisition_channel', 'cluster']).size().reset_index(name='count')

plt.figure(figsize=(10, 4))
sns.barplot(data=channel_perf, x='acquisition_channel', y='avg_spend')
plt.title('Average spend per channel')
plt.show()

pivot = cluster_mix.pivot(index='acquisition_channel', columns='cluster', values='count').fillna(0)
(pivot.T / pivot.sum(axis=1)).T.plot(kind='bar', stacked=True, figsize=(10, 4))
plt.ylabel('Cluster share')
plt.title('Cluster composition by channel')
plt.show()

if not trans_counts.empty:
    for channel, channel_df in full_grid.groupby('acquisition_channel'):
        matrix = transition_matrix(channel_df, weighted=False)
        if matrix.empty:
            continue
        plt.figure(figsize=(6, 4))
        sns.heatmap(matrix, annot=True, fmt='.0f', cmap='Greens')
        plt.title(f'Transitions — {channel}')
        plt.show()


## 11. Бізнес-інсайти
- **Траєкторії**: матриці переходів показують, наскільки часто virtual → pause → tv та інші маршрути; це визначає вікна реактивації.
- **Типи гравців**: кластери приблизно відповідають `Steady Spenders`, `Cross-market Switchers`, `Sprinters`, `Entertainment-first`.
- **Життєвий цикл**: метрики streak/gap/zero_share підказують, коли користувачі «остивають» та коли їх варто активувати промо.
- **Прогноз**: моделі LightGBM (якщо доступні) оцінюють сектор і spend наступного дня, а також cross-sector activation.
- **Аномалії**: IsolationForest/ruptures/HMM дозволяють виявити різкі зміни та потенційно ризикових користувачів.
- **Канали**: агреговані метрики показують, які acquisition-канали дають вищий LTV, довші активні періоди та багатші переходи між секторами.

Подальші кроки: інтегрувати реальні дані, тюнити моделі, підготувати рекомендації для retention та крос-продажів.
